In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import datetime, timedelta
import plotly.io as pio
import pytz
import plotly.graph_objects as go
import webbrowser

In [2]:
apps_df = pd.read_csv("google_play_store_dataset.csv")
reviews_df = pd.read_csv("App_Sentiment_Analysis.csv")

In [3]:
apps_df=apps_df.dropna(subset=['Rating'])

In [4]:
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)

C:\Users\pawan\AppData\Local\Temp\ipykernel_12624\4000964054.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)


In [5]:
apps_df.drop_duplicates(inplace=True)
reviews_df.dropna(subset=['Translated_Review'],inplace=True)
apps_df = apps_df[apps_df['Rating']<=5]
reviews_df.dropna(subset=['Translated_Review'], inplace = True)
apps_df['Installs']= apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)
apps_df['Price'] = apps_df['Price'].str.replace('$','').astype(float)

In [6]:
merged_df = pd.merge(apps_df,reviews_df,on = 'App',how = 'inner')

In [7]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'K' in size:
        return float(size.replace('K',''))/1024
    else:
        return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)

In [8]:
apps_df['Reviews'] = apps_df['Reviews'].astype(float)

In [9]:
apps_df['Log_Installs']=np.log(apps_df['Installs'])
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [10]:
def rating_group(rating):
    if rating >=4:
        return 'Top rated app'
    elif rating >=3:
        return 'Above Average app'
    elif rating >=2:
        return 'Average app'
    else:
        return 'Below Average app'
apps_df['Rating_Group']=apps_df['Rating'].apply(rating_group)

In [11]:
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [12]:
sia = SentimentIntensityAnalyzer()
reviews_df['Sentiment_Score']=reviews_df['Translated_Review'].apply(lambda x : sia.polarity_scores(str(x))['compound'])

In [13]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

<IPython.core.display.Javascript object>

In [14]:
plot_containers=""

In [15]:
def save_plot_as_html(fig,filename,insight):
    global plot_containers
    filepath=os.path.join(html_files_path,filename)
    html_content=pio.to_html(fig,full_html=False,include_plotlyjs='inline')
    plot_containers+=f"""
        <div class="plot-container"id="{filename}" onclick = "openPlot('{filename}')">
            <div class="plot">{html_content}</div>
            <div class="insights">{insight}</div>
        </div>
    """
    fig.write_html(filepath,full_html=False,include_plotlyjs='inline')

In [16]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [17]:
# ---- Get IST Time ----
ist = pytz.timezone("Asia/Kolkata")
ist_now = datetime.now(ist)

In [18]:
apps_df['Sentiment_Subjectivity'] = reviews_df['Sentiment_Subjectivity']

In [23]:
apps_df1 = apps_df.copy()

In [24]:
apps_df1.head()


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,0.533333
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,13.122363,6.874198,Above Average app,0.0,0.288462
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,NaN
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,0.875000
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,0.300000


In [29]:
# ---- Time Restriction (3 PM – 5 PM IST) ----
if not (15 <= ist_now.hour < 17):
    print("Graph only visible between 3 PM and 5 PM IST")
else:

    # ---- Data Cleaning ----
    apps_df1["Rating"] = pd.to_numeric(apps_df1["Rating"], errors="coerce")
    apps_df1["Reviews"] = pd.to_numeric(apps_df1["Reviews"], errors="coerce")
    apps_df1["Installs"] = pd.to_numeric(apps_df1["Installs"], errors="coerce")
    apps_df1["Size"] = pd.to_numeric(apps_df1["Size"], errors="coerce")

    # Convert date column
    apps_df1["Last Updated"] = pd.to_datetime(apps_df1["Last Updated"], errors="coerce")

    # ---- Apply Filters ----
    filtered_df = apps_df1[
        (apps_df1["Rating"] >= 4.0) &
        (apps_df1["Size"] >= 10) &
        (apps_df1["Last Updated"].dt.month == 1)   # January
    ]

    # ---- Top 10 Categories by Total Installs ----
    top_categories = (
        filtered_df.groupby("Category")["Installs"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .index
    )

    top_df = filtered_df[
        filtered_df["Category"].isin(top_categories)
    ]

    # ---- Aggregate Data ----
    grouped_df = (
        top_df.groupby("Category")
        .agg(
            Avg_Rating=("Rating", "mean"),
            Total_Reviews=("Reviews", "sum")
        )
        .reset_index()
    )

    # ---- Create Grouped Bar Chart ----
    figure1 = go.Figure()

    figure.add_trace(go.Bar(
        x=grouped_df["Category"],
        y=grouped_df["Avg_Rating"],
        name="Average Rating"
    ))

    figure1.add_trace(go.Bar(
        x=grouped_df["Category"],
        y=grouped_df["Total_Reviews"],
        name="Total Reviews"
    ))

    # ---- Layout Styling ----
    figure1.update_layout(
        title="Top 10 App Categories: Average Rating vs Total Reviews",
        xaxis_title="App Category",
        yaxis_title="Value",
        barmode="group",
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure1,
        "GroupedBar_Top10_Category_Rating_Reviews.html",
        "This grouped bar chart compares the average rating and total review count for the top 10 app categories ranked by installs. Only categories meeting strict quality filters such as rating ≥ 4.0, size ≥ 10 MB, and apps updated in January are included."
    )

Graph only visible between 3 PM and 5 PM IST


In [28]:
apps_df2 = apps_df.copy()

In [33]:
if not (18 <= ist_now.hour < 20):
    print("Graph only visible between 6 PM and 8 PM IST")
else:

    # ---- Exclude categories starting with A, C, G, S ----
    filtered_df = apps_df2[
        ~apps_df2["Category"].str.startswith(('A', 'C', 'G', 'S'))
    ]

    # ---- Groupby Category ----
    category_installs = filtered_df.groupby("Category").agg(
        Total_Installs=('Installs', 'sum')
    ).reset_index()

    # ---- Top 5 Categories by Installs ----
    top_5 = category_installs.sort_values(
        "Total_Installs",
        ascending=False
    ).head(5)

    # ---- Highlight categories with installs > 1M ----
    top_5["Highlight"] = top_5["Total_Installs"].apply(
        lambda x: "Above 1M" if x > 1_000_000 else "Below 1M"
    )

    # ---- Create Choropleth Map ----
    figure2 = go.Figure(data=go.Choropleth(
        locations=top_5['Category'],   # Using Category as region placeholder
        locationmode='country names',  # Required format for Choropleth
        z=top_5['Total_Installs'],
        text=top_5['Highlight'],
        colorscale='Viridis',
        colorbar_title="Total Installs"
    ))

    figure2.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title='Top 5 App Categories (Excluding A, C, G, S) - Global Installs',
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save as HTML ----
    save_plot_as_html(figure2,"Global Installs Choropleth.html","The visualization shows that the leading app categories (excluding A, C, G, S) dominate global installs, with highlighted segments surpassing 1 million installs. This indicates strong user demand concentrated in a few high-performing categories.")

Graph only visible between 6 PM and 8 PM IST


In [32]:
apps_df3 = apps_df.copy()

In [35]:
if not (13 <= ist_now.hour < 14):
    print("Graph only visible between 1 PM and 2 PM IST")
else:

    # ---- Data Cleaning ----
    
    # Convert Installs & Revenue to numeric (if needed)
    apps_df3["Installs"] = pd.to_numeric(apps_df3["Installs"], errors="coerce")
    apps_df3["Revenue"] = pd.to_numeric(apps_df3["Revenue"], errors="coerce")

    # Extract numeric android version (example: '4.1 and up')
    apps_df3["Android_Version_Num"] = (
        apps_df3["Android Ver"]
        .str.extract(r'(\d+\.?\d*)')[0]
        .astype(float)
    )


    # ---- Apply All Filters ----
    filtered_df = apps_df3[
        (apps_df3["Installs"] >= 10000) &
        (apps_df3["Revenue"] >= 10000) &
        (apps_df3["Android_Version_Num"] > 4.0) &
        (apps_df3["Size"] > 15) &
        (apps_df3["Content Rating"] == "Everyone") &
        (apps_df3["App"].str.len() <= 30)
    ]

    # ---- Group by Category & Type ----
    grouped_df = filtered_df.groupby(
        ["Category", "Type"]
    ).agg(
        Avg_Installs=("Installs", "mean"),
        Avg_Revenue=("Revenue", "mean")
    ).reset_index()

    # ---- Top 3 Categories by Avg Installs ----
    top_3_categories = (
        grouped_df.groupby("Category")["Avg_Installs"]
        .mean()
        .sort_values(ascending=False)
        .head(3)
        .index
    )

    top_3_df = grouped_df[
        grouped_df["Category"].isin(top_3_categories)
    ]

    # ---- Create Dual Axis Chart ----
    figure3 = go.Figure()

    # Avg Installs (Primary Y-Axis)
    figure3.add_trace(go.Bar(
        x=top_3_df["Category"] + " - " + top_3_df["Type"],
        y=top_3_df["Avg_Installs"],
        name="Average Installs",
        yaxis="y1"
    ))

    # Avg Revenue (Secondary Y-Axis)
    figure3.add_trace(go.Scatter(
        x=top_3_df["Category"] + " - " + top_3_df["Type"],
        y=top_3_df["Avg_Revenue"],
        name="Average Revenue",
        yaxis="y2",
        mode="lines+markers"
    ))

    # ---- Layout Styling ----
    figure3.update_layout(
        title="Top 3 Categories: Avg Installs vs Revenue (Free vs Paid)",
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        yaxis=dict(
            title="Average Installs",
            showgrid=False
        ),
        yaxis2=dict(
            title="Average Revenue",
            overlaying="y",
            side="right",
            showgrid=False
        ),
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure3,
        "Dual_Axis_Top3_Category_Comparison.html",
        "This dual-axis visualization compares average installs and revenue for Free vs Paid apps within the top 3 categories. Only apps meeting strict quality filters are included. The graph highlights monetization differences between Free and Paid models under high-performance app segments."
    )

Graph only visible between 1 PM and 2 PM IST


In [36]:
apps_df4 = apps_df.copy()

In [38]:
# ---- Time Restriction (6 PM – 9 PM IST) ----
if not (18 <= ist_now.hour < 21):
    print("Graph only visible between 6 PM and 9 PM IST")
else:

    # ---- Data Cleaning ----

    # Convert Installs & Reviews to numeric
    apps_df4["Installs"] = pd.to_numeric(apps_df4["Installs"], errors="coerce")
    apps_df4["Reviews"] = pd.to_numeric(apps_df4["Reviews"], errors="coerce")

    # Convert Date column to datetime (assuming column name is Last Updated)
    apps_df4["Last Updated"] = pd.to_datetime(apps_df4["Last Updated"], errors="coerce")

    # Extract Month-Year for time series
    apps_df4["Month"] = apps_df4["Last Updated"].dt.to_period("M").dt.to_timestamp()

    # ---- Category Translation ----
    category_translation = {
        "Beauty": "सौंदर्य",     # Hindi
        "Business": "வணிகம்",    # Tamil
        "Dating": "Dating"      # will change to German below
    }

    apps_df4["Category"] = apps_df4["Category"].replace(category_translation)

    # German translation
    apps_df4.loc[apps_df["Category"] == "Dating", "Category"] = "Partnersuche"

    # ---- Apply Filters ----
    filtered_df = apps_df4[
        (~apps_df4["App"].str.lower().str.startswith(("x", "y", "z"))) &  # app name not start with x,y,z
        (~apps_df4["App"].str.contains("S", case=False, na=False)) &      # no S in name
        (apps_df4["Category"].str.startswith(("E", "C", "B", "स", "வ", "P"))) &  # categories starting with allowed letters
        (apps_df4["Reviews"] > 500)
    ]

    # ---- Aggregate Installs Over Time ----
    grouped_df = (
        filtered_df
        .groupby(["Month", "Category"])
        .agg(Total_Installs=("Installs", "sum"))
        .reset_index()
        .sort_values("Month")
    )

    # ---- Calculate Month-over-Month Growth ----
    grouped_df["MoM_Growth"] = (
        grouped_df.groupby("Category")["Total_Installs"]
        .pct_change()
    )

    # ---- Create Plotly Figure ----
    figure4 = go.Figure()

    categories = grouped_df["Category"].unique()

    for cat in categories:
        cat_df = grouped_df[grouped_df["Category"] == cat]

        # Line chart
        figure4.add_trace(go.Scatter(
            x=cat_df["Month"],
            y=cat_df["Total_Installs"],
            mode="lines+markers",
            name=cat
        ))

        # Highlight >20% growth
        high_growth = cat_df[cat_df["MoM_Growth"] > 0.20]

        figure4.add_trace(go.Scatter(
            x=high_growth["Month"],
            y=high_growth["Total_Installs"],
            fill="tozeroy",
            mode="none",
            name=f"{cat} Growth >20%",
            opacity=0.2
        ))

    # ---- Layout Styling ----
    figure4.update_layout(
        title="Monthly Trend of Total Installs by App Category",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure4,
        "TimeSeries_App_Installs_Growth.html",
        "This time-series visualization shows the trend of total installs over time segmented by app category. Areas shaded under the curve highlight periods where installs increased more than 20% month-over-month. Category names are translated into Hindi, Tamil, and German for multilingual representation. Only apps meeting strict filtering criteria are included."
    )

Graph only visible between 6 PM and 9 PM IST


In [39]:
apps_df5 = apps_df.copy()

In [42]:
# ---- Time Restriction (5 PM – 7 PM IST) ----
if not (17 <= ist_now.hour < 19):
    print("Graph only visible between 5 PM and 7 PM IST")
else:

    # ---- Data Cleaning ----

    apps_df5["Rating"] = pd.to_numeric(apps_df5["Rating"], errors="coerce")
    apps_df5["Reviews"] = pd.to_numeric(apps_df5["Reviews"], errors="coerce")
    apps_df5["Installs"] = pd.to_numeric(apps_df5["Installs"], errors="coerce")
    apps_df5["Size"] = pd.to_numeric(apps_df5["Size"], errors="coerce")
    apps_df5["Sentiment_Subjectivity"] = pd.to_numeric(
        apps_df5["Sentiment_Subjectivity"], errors="coerce"
    )

    # ---- Category Translation ----
    category_translation = {
        "Beauty": "सौंदर्य",        # Hindi
        "Business": "வணிகம்",       # Tamil
        "Dating": "Partnersuche"    # German
    }

    apps_df5["Category"] = apps_df5["Category"].replace(category_translation)

    # ---- Allowed Categories ----
    allowed_categories = [
        "Game",
        "सौंदर्य",        # Beauty translated
        "வணிகம்",         # Business translated
        "Comics",
        "Communication",
        "Partnersuche",   # Dating translated
        "Entertainment",
        "Social",
        "Events"
    ]

    # ---- Apply Filters ----
    filtered_df = apps_df5[
        (apps_df5["Rating"] > 3.5) &
        (apps_df5["Category"].isin(allowed_categories)) &
        (apps_df5["Reviews"] > 500) &
        (~apps_df5["App"].str.contains("S", case=False, na=False)) &
        (apps_df5["Sentiment_Subjectivity"] > 0.5) &
        (apps_df5["Installs"] > 50000)
    ]

    # ---- Bubble Chart ----
    figure5 = px.scatter(
        filtered_df,
        x="Size",
        y="Rating",
        size="Installs",
        color="Category",
        hover_name="App",
        title="App Size vs Rating (Bubble size = Installs)"
    )

    # ---- Highlight Game Category in Pink ----
    for trace in figure.data:
        if trace.name == "Game":
            trace.marker.color = "pink"

    # ---- Layout Styling ----
    figure5.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        xaxis_title="App Size (MB)",
        yaxis_title="Average Rating",
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure5,
        "Bubble_App_Size_vs_Rating.html",
        "This bubble chart visualizes the relationship between app size and average rating. Bubble size represents installs. Only high-quality apps meeting strict filtering criteria are shown. Categories include Game, Beauty, Business, Comics, Communication, Dating, Entertainment, Social, and Events with multilingual category translations."
    )

Graph only visible between 5 PM and 7 PM IST


In [43]:
apps_df6 = apps_df.copy()

In [46]:
# ---- Time Restriction (4 PM – 6 PM IST) ----
if not (16 <= ist_now.hour < 18):
    print("Graph only visible between 4 PM and 6 PM IST")
else:

    # ---- Data Cleaning ----
    apps_df6["Rating"] = pd.to_numeric(apps_df6["Rating"], errors="coerce")
    apps_df6["Reviews"] = pd.to_numeric(apps_df6["Reviews"], errors="coerce")
    apps_df6["Installs"] = pd.to_numeric(apps_df6["Installs"], errors="coerce")
    apps_df6["Size"] = pd.to_numeric(apps_df6["Size"], errors="coerce")

    # Convert date
    apps_df6["Last Updated"] = pd.to_datetime(apps_df6["Last Updated"], errors="coerce")

    # Month column for time series
    apps_df6["Month"] = apps_df6["Last Updated"].dt.to_period("M").dt.to_timestamp()

    # ---- Category Translation for Legend ----
    category_translation = {
        "Travel & Local": "Voyage et Local",   # French
        "Productivity": "Productividad",       # Spanish
        "Photography": "写真"                  # Japanese
    }

    apps_df6["Category"] = apps_df6["Category"].replace(category_translation)

    # ---- Apply Filters ----
    filtered_df = apps_df6[
        (apps_df6["Rating"] >= 4.2) &
        (apps_df6["Reviews"] > 1000) &
        (apps_df6["Size"].between(20, 80)) &
        (~apps_df6["App"].str.contains(r"\d", na=False)) &  # no numbers in app name
        (apps_df6["Category"].str.startswith(("T", "P", "V", "P", "写")))  # categories starting with T or P (plus translated)
    ]

    # ---- Aggregate Installs Over Time ----
    grouped_df = (
        filtered_df
        .groupby(["Month", "Category"])
        .agg(Total_Installs=("Installs", "sum"))
        .reset_index()
        .sort_values("Month")
    )

    # ---- Calculate MoM Growth ----
    grouped_df["MoM_Growth"] = (
        grouped_df.groupby("Category")["Total_Installs"]
        .pct_change()
    )

    # ---- Create Stacked Area Chart ----
    figure6 = go.Figure()

    categories = grouped_df["Category"].unique()

    for cat in categories:

        cat_df = grouped_df[grouped_df["Category"] == cat]

        figure6.add_trace(go.Scatter(
            x=cat_df["Month"],
            y=cat_df["Total_Installs"],
            stackgroup="one",
            mode="lines",
            name=cat,
            line=dict(width=2)
        ))

        # Highlight months where installs grew >25%
        high_growth = cat_df[cat_df["MoM_Growth"] > 0.25]

        figure6.add_trace(go.Scatter(
            x=high_growth["Month"],
            y=high_growth["Total_Installs"],
            mode="markers",
            marker=dict(size=12),
            name=f"{cat} Growth >25%"
        ))

    # ---- Layout Styling ----
    figure6.update_layout(
        title="Cumulative App Installs Over Time by Category",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure6,
        "Stacked_Area_App_Installs.html",
        "This stacked area chart visualizes cumulative installs over time for each app category. Only high-quality apps with strong ratings, reviews, and specific size constraints are included. Categories such as Travel & Local, Productivity, and Photography are translated into French, Spanish, and Japanese. Months where installs increased by more than 25% month-over-month are highlighted."
    )

Graph only visible between 4 PM and 6 PM IST


In [47]:
plot_containers_split=plot_containers.split('</div>')

In [48]:
if len(plot_containers_split)>1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [49]:
dashboard_html="""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width, initial-scale-1.0">
    <title>Google PlayStore Analytics</title>
    <style>
        body{{
            font-family:Arial.sans-serif;
            background-color:#F0FFFF;
            color:0;
            padding:0;
        }}
        .header{{
            display:flex;
            align-items:center;
            justify-content:center;
            padding:20px;
            background-color=#444
        }}
        .header img{{
            margin:0 10px;
            height:50px;
        }}
        .container{{
            display:flex;
            flex-wrap:wrap;
            justify-content:center;
            padding:20px;
        }}
        .plot-container{{
            border:2px solid #555
            margin:10px;
            padding:10px;
            width:{plot_width}px;
            height:{plot_height}px;
            overflow:hidden;
            position:relative;
            cursor:pointer;
        }}
        .insights{{
            display:none;
            position:absolute;
            right:10px;
            top:10px;
            background-color:rgba(0,0,0,0.7);
            padding:5px;
            border-radius:5px;
            color:#fff;
        }}
        .plot-container:hover .insights{{
            display:block;
        }}
    </style>
    <script>
        function.openPlot(filename){{
            window.open(filename,'_blank');
        }}
    </script>
    </head>
    <body>
        <div class="header">
            <img src="https://static.vecteezy.com/system/resources/thumbnails/028/667/072/small_2x/google-logo-icon-symbol-free-png.png" alt="Google Logo">
            <h1>Google PlayStore Reviews Analytics</h1>
            <img src="https://freelogopng.com/images/all_img/1664287128google-play-store-logo-png.png" alt="Google PlayStore Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
"""

In [50]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [51]:
dashboard_path=os.path.join(html_files_path,"web page.html")

<IPython.core.display.Javascript object>

In [52]:
with open(dashboard_path,"w",encoding="utf-8") as f:
    f.write(final_html)

In [56]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

<IPython.core.display.Javascript object>

True